In [6]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [25]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1)

        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)

        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(in_features=16 * 5 * 5, out_features=120)

        self.fc2 = nn.Linear(in_features=120, out_features=84)

        self.fc3 = nn.Linear(in_features=84, out_features=10)

    def forward(self, x):

        x = self.pool1(torch.tanh(self.conv1(x)))
        x = self.pool2(torch.tanh(self.conv2(x)))

        x = x.view(-1, 16 * 5 * 5)

        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)

        return x

In [8]:
model = LeNet5()

In [9]:
loss_fn =  nn.CrossEntropyLoss()
optimiser =  torch.optim.Adam(model.parameters(),lr=0.001)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set,batch_size=1,shuffle=True)
test_loader = DataLoader(test_set,batch_size=1,shuffle=True)

In [11]:
print(model.parameters)
print([p.numel() for p in model.parameters()])
params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {params:,}")

<bound method Module.parameters of LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>
[150, 6, 2400, 16, 48000, 120, 10080, 84, 840, 10]
Total Parameters: 61,706


In [12]:
from ptflops import get_model_complexity_info
macs, params = get_model_complexity_info(model, (1, 32, 32), as_strings=True, print_per_layer_stat=False)
print(f"Computational complexity: {macs}")
print(params)

Computational complexity: 435.65 KMac
61.71 k


In [13]:
macs, params = get_model_complexity_info(model, (1, 32, 32), as_strings=True, print_per_layer_stat=True)

LeNet5(
  61.71 k, 100.000% Params, 429.34 KMac, 98.553% MACs, 
  (conv1): Conv2d(156, 0.253% Params, 122.3 KMac, 28.074% MACs, 1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(0, 0.000% Params, 4.7 KMac, 1.080% MACs, kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(2.42 k, 3.915% Params, 241.6 KMac, 55.458% MACs, 6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(0, 0.000% Params, 1.6 KMac, 0.367% MACs, kernel_size=2, stride=2, padding=0)
  (fc1): Linear(48.12 k, 77.983% Params, 48.12 KMac, 11.046% MACs, in_features=400, out_features=120, bias=True)
  (fc2): Linear(10.16 k, 16.472% Params, 10.16 KMac, 2.333% MACs, in_features=120, out_features=84, bias=True)
  (fc3): Linear(850, 1.377% Params, 850.0 Mac, 0.195% MACs, in_features=84, out_features=10, bias=True)
)


In [14]:
print(next(iter(train_loader)))

[tensor([[[[-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242],
          ...,
          [-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242],
          [-0.4242, -0.4242, -0.4242,  ..., -0.4242, -0.4242, -0.4242]]]]), tensor([4])]


In [15]:
"""from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

pbar = tqdm(train_loader)

all_preds = []
all_labels = []

for image,label in pbar:

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_Metrics.add_scalar("Loss/train - LeNet-5 IT599 Ben", loss.item(), global_i)
    LeNet5_Metrics.add_scalar("Accuracy/train - LeNet-5 IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.numpy())
    all_labels.extend(label.numpy())

    global_i += 1

LeNet5_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")"""

'from tqdm.notebook import tqdm\nimport time\nfrom torch.utils.tensorboard import SummaryWriter\nLeNet5_Metrics = SummaryWriter()\n\nlatency_per_image = []\nglobal_i = 0\n\npbar = tqdm(train_loader)\n\nall_preds = []\nall_labels = []\n\nfor image,label in pbar:\n\n    start_time = time.time()\n\n    optimiser.zero_grad()\n\n    y_pred =  model(image)\n\n    loss = loss_fn(y_pred,label)\n\n    loss.backward()\n\n    optimiser.step()\n\n    _, predicted = torch.max(y_pred.data, 1)\n    correct = (predicted == label).sum().item()\n    accuracy = correct / label.size(0)\n\n    end_time = time.time()\n\n    latency_per_image.append(end_time - start_time)\n\n    LeNet5_Metrics.add_scalar("Loss/train - LeNet-5 IT599 Ben", loss.item(), global_i)\n    LeNet5_Metrics.add_scalar("Accuracy/train - LeNet-5 IT599 Ben", accuracy, global_i)\n\n    all_preds.extend(predicted.numpy())\n    all_labels.extend(label.numpy())\n\n    global_i += 1\n\nLeNet5_Metrics.close()\n\navg_latency_batch = sum(latency_

In [16]:
#python3 -m tensorboard.main --logdir="/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/LeNet-5/runs"

In [17]:
"""from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds))"""

'from sklearn.metrics import classification_report, confusion_matrix\n\nprint(classification_report(all_labels, all_preds))'

In [26]:
import torchvision
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

In [27]:
model_CIFAR = LeNet5()

loss_fn = nn.CrossEntropyLoss()

optimiser = torch.optim.Adam(model_CIFAR.parameters(), lr=0.001)

from torch.utils.data import DataLoader

train_loader = DataLoader(trainset, batch_size=1, shuffle=True)
test_loader = DataLoader(testset, batch_size=1, shuffle=True)

In [31]:
print(model_CIFAR.parameters)
print([p.numel() for p in model_CIFAR.parameters()])
params = sum(p.numel() for p in model_CIFAR.parameters())
print(f"Total Parameters: {params:,}")

<bound method Module.parameters of LeNet5(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>
[450, 6, 2400, 16, 48000, 120, 10080, 84, 840, 10]
Total Parameters: 62,006


In [34]:
from ptflops import get_model_complexity_info
macs, params = get_model_complexity_info(model_CIFAR, (3, 32, 32), as_strings=True, print_per_layer_stat=False)
print(f"Computational complexity: {macs}")
print(params)

Computational complexity: 670.85 KMac
62.01 k


In [28]:
print(next(iter(train_loader)))

[tensor([[[[2.8215, 2.8215, 2.8215,  ..., 2.8215, 2.8215, 2.8215],
          [2.8215, 2.8088, 2.8088,  ..., 2.7706, 2.7960, 2.8088],
          [2.8215, 2.8215, 2.8215,  ..., 2.2487, 2.8215, 2.8088],
          ...,
          [1.2050, 1.2177, 1.2177,  ..., 1.2814, 1.3068, 1.3577],
          [1.2177, 1.2177, 1.1795,  ..., 1.2177, 1.2432, 1.2941],
          [1.2177, 1.1923, 1.1795,  ..., 1.1541, 1.1923, 1.2559]],

         [[2.8215, 2.8215, 2.8215,  ..., 2.8215, 2.8215, 2.8215],
          [2.8215, 2.8088, 2.8088,  ..., 2.7706, 2.7960, 2.8088],
          [2.8215, 2.8215, 2.8215,  ..., 2.2487, 2.8215, 2.8088],
          ...,
          [1.2050, 1.2177, 1.2177,  ..., 1.2814, 1.3068, 1.3577],
          [1.2177, 1.2177, 1.1795,  ..., 1.2177, 1.2432, 1.2941],
          [1.2177, 1.1923, 1.1795,  ..., 1.1541, 1.1923, 1.2559]],

         [[2.8215, 2.8215, 2.8215,  ..., 2.8215, 2.8215, 2.8215],
          [2.8215, 2.8088, 2.8088,  ..., 2.7706, 2.7960, 2.8088],
          [2.8215, 2.8215, 2.8215,  ..., 

In [29]:
from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_CIFAR_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

pbar = tqdm(train_loader)

all_preds = []
all_labels = []

for image,label in pbar:

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model_CIFAR(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_CIFAR_Metrics.add_scalar("Loss/train - LeNet_5_CIFAR IT599 Ben", loss.item(), global_i)
    LeNet5_CIFAR_Metrics.add_scalar("Accuracy/train - LeNet_5_CIFAR IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.numpy())
    all_labels.extend(label.numpy())

    global_i += 1

LeNet5_CIFAR_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")

  0%|          | 0/50000 [00:00<?, ?it/s]

Latency: 1.42 ms | Throughput: 703.58 items/sec


In [35]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.39      0.41      0.40      5000
           1       0.43      0.47      0.45      5000
           2       0.26      0.18      0.22      5000
           3       0.25      0.21      0.23      5000
           4       0.29      0.27      0.28      5000
           5       0.27      0.25      0.26      5000
           6       0.36      0.45      0.40      5000
           7       0.40      0.44      0.42      5000
           8       0.41      0.45      0.43      5000
           9       0.38      0.38      0.38      5000

    accuracy                           0.35     50000
   macro avg       0.34      0.35      0.35     50000
weighted avg       0.34      0.35      0.35     50000

